<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
M28 | Gradio UI für Agenten
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul gradio

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M27-Gradio-UI"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---

Gradio ist ein Python-Framework für schnelle Web-UIs — ideal um LangGraph-Agenten
interaktiv erlebbar zu machen, ohne HTML/CSS/JS schreiben zu müssen.

| Gradio-Komponente | Einsatz im Agenten-Kontext |
|-------------------|---------------------------|
| `gr.ChatInterface` | Einfachste Chat-UI: eine Funktion, fertig |
| `gr.Blocks` | Mehrere Panels, Tool-Log, eigene Layouts |
| `gr.State` | Sitzungsstatus (z.B. Thread-ID, ausstehende Calls) |
| Generator / `yield` | Streaming: Antwort erscheint Token für Token |
| Buttons (sichtbar/versteckt) | Human-in-the-Loop: Approve / Reject |


# 2 | gr.ChatInterface — Einfaches Agent-UI
---

`gr.ChatInterface` ist die schnellste Variante: eine Python-Funktion `fn(message, history) -> str`
wird automatisch als Chat-UI gerendert.

```python
import gradio as gr

def chat_fn(message: str, history: list) -> str:
    return agent.invoke(...)

gr.ChatInterface(fn=chat_fn).launch()
```

| Parameter | Bedeutung |
|-----------|----------|
| `fn` | Callback-Funktion `(message, history) → str` |
| `title` | Überschrift der UI |
| `examples` | Klickbare Beispiel-Inputs |
| `type="messages"` | Modernes Nachrichtenformat (Gradio 4+) |
| `share=True` | Öffentliche URL (Colab/Remote) |

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart: ChatInterface Architektur</font> </br></p>

diagram = '''
%%{init: {'theme':'dark'}}%%
flowchart LR
    U(["👤 User"])
    CI["gr.ChatInterface\nchat_fn(message, history)"]
    A["ReAct Agent\ngpt-4o-mini"]
    T["Tools\n(addiere, multipliziere, ...)"]

    U -->|Eingabe| CI
    CI -->|message| A
    A -->|Tool Call| T
    T -->|Ergebnis| A
    A -->|str| CI
    CI -->|Antwort| U

    style CI  fill:#1565C0,color:#fff
    style A   fill:#2E7D32,color:#fff
    style T   fill:#37474F,color:#fff
    style U   fill:#E65100,color:#fff
'''
mermaid(diagram, width=1100)

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.prebuilt import create_react_agent

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

@tool
def addiere(a: float, b: float) -> float:
    '''Addiert zwei Zahlen.'''
    return a + b

@tool
def multipliziere(a: float, b: float) -> float:
    '''Multipliziert zwei Zahlen.'''
    return a * b

@tool
def berechne_prozent(wert: float, prozent: float) -> float:
    '''Berechnet prozent% von wert.'''
    return round(wert * prozent / 100, 2)

tools_liste = [addiere, multipliziere, berechne_prozent]
agent = create_react_agent(llm, tools=tools_liste)
print("✅ LLM (gpt-4o-mini) und ReAct-Agent mit 3 Tools bereit")

In [ ]:
import gradio as gr

def chat_fn(message: str, history: list) -> str:
    '''Gradio-Callback: übergibt Nachricht an Agenten, gibt Antwort zurück.'''
    result = agent.invoke({"messages": [HumanMessage(message)]})
    return result["messages"][-1].content

demo_basic = gr.ChatInterface(
    fn=chat_fn,
    title="🤖 Mathe-Agent",
    description="Stelle Rechenaufgaben — der Agent löst sie mit Tools.",
    examples=[
        "Was ist 127 × 8?",
        "Berechne 15% von 240 Euro.",
        "Addiere 1.234 und 5.678, dann multipliziere mit 2.",
    ],
    type="messages",
)
demo_basic.launch(quiet=True)

# 3 | gr.Blocks — Chat mit Tool-Visualisierung
---

`gr.Blocks` erlaubt freies Layout mit mehreren Panels. Hier: Chat links, Tool-Log rechts.
Der Agent wird nach jedem Aufruf analysiert — alle Tool-Calls werden sichtbar gemacht.

```
┌─────────────────────────┬──────────────────┐
│  💬 Chat                │  📋 Tool-Aufrufe │
│                         │                  │
│  User: Was ist 15×8?    │  ▶ multipliziere │
│  Agent: Das Ergebnis    │    (a=15, b=8)   │
│         ist 120.        │                  │
├─────────────────────────┴──────────────────┤
│  [ Eingabe...                    ] [Senden] │
└────────────────────────────────────────────┘
```

In [ ]:
import gradio as gr

def verarbeite(nachricht: str, verlauf: list):
    '''Führt Agenten aus, extrahiert Tool-Aufrufe für den Log.'''
    result = agent.invoke({"messages": [HumanMessage(nachricht)]})

    # Tool-Aufrufe aus LangGraph-Messages extrahieren
    tool_log_lines = []
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                args_str = ", ".join(f"{k}={v}" for k, v in tc["args"].items())
                tool_log_lines.append(f"▶ {tc['name']}({args_str})")

    antwort = result["messages"][-1].content
    verlauf.append({"role": "user",      "content": nachricht})
    verlauf.append({"role": "assistant", "content": antwort})
    tool_text = "\n".join(tool_log_lines) if tool_log_lines else "(keine Tools aufgerufen)"
    return "", verlauf, tool_text

with gr.Blocks(title="Agent mit Tool-Log", theme=gr.themes.Soft()) as demo_blocks:
    gr.Markdown("## 🔧 Agent mit Tool-Visualisierung")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(type="messages", label="💬 Chat", height=300)
        with gr.Column(scale=1):
            tool_box = gr.Textbox(
                label="📋 Tool-Aufrufe", lines=10,
                interactive=False, placeholder="Noch keine Tool-Aufrufe..."
            )
    with gr.Row():
        eingabe = gr.Textbox(placeholder="Stelle eine Rechenaufgabe...",
                             label="Eingabe", scale=4)
        senden  = gr.Button("Senden", variant="primary", scale=1)

    senden.click(verarbeite, [eingabe, chatbot], [eingabe, chatbot, tool_box])
    eingabe.submit(verarbeite, [eingabe, chatbot], [eingabe, chatbot, tool_box])

demo_blocks.launch(quiet=True)

# 4 | Streaming — Agent-Antworten in Echtzeit
---

LangGraph unterstützt `graph.stream()` — damit lassen sich Antworten Token für Token
an Gradio weiterleiten. Die Callback-Funktion wird zum **Generator** mit `yield`.

```python
def stream_fn(message, history):
    antwort = ""
    for chunk in agent.stream({"messages": [HumanMessage(message)]}):
        if "agent" in chunk:
            for msg in chunk["agent"]["messages"]:
                if msg.content:
                    antwort += msg.content
                    yield antwort   # ← Gradio zeigt Zwischenstand
```

> **Hinweis:** `stream()` liefert Chunks pro Node — nicht pro Token.
> Für echtes Token-Streaming ist `astream_events()` nötig (fortgeschritten).

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  sequenceDiagram: Streaming-Flow</font> </br></p>

diagram = '''

sequenceDiagram
    participant U as 👤 User
    participant G as gr.ChatInterface
    participant LG as LangGraph
    participant T as Tools

    U->>G: Frage
    G->>LG: stream({"messages": [...]})
    loop Chunk für Chunk
        LG-->>G: {"agent": {"messages": [...]}}
        G-->>U: yield partial_antwort
    end
    LG->>T: Tool Call
    T-->>LG: Ergebnis
    LG-->>G: {"agent": {"messages": [finale Antwort]}}
    G-->>U: yield finale_antwort
'''
mermaid(diagram, width=900)

In [ ]:
import gradio as gr

def stream_agent(message: str, history: list):
    '''Generator: streamt Agent-Antwort Chunk für Chunk via LangGraph.stream().'''
    yield "⏳ Agent arbeitet..."
    antwort = ""
    for chunk in agent.stream({"messages": [HumanMessage(message)]}):
        if "agent" in chunk:
            for msg in chunk["agent"].get("messages", []):
                # Nur Text-Antworten streamen (keine Tool-Call-Messages)
                if hasattr(msg, "content") and msg.content:
                    if not (hasattr(msg, "tool_calls") and msg.tool_calls):
                        antwort += msg.content
                        yield antwort
    # Fallback: direkte Antwort wenn kein Stream-Content
    if not antwort:
        result = agent.invoke({"messages": [HumanMessage(message)]})
        yield result["messages"][-1].content

demo_stream = gr.ChatInterface(
    fn=stream_agent,
    title="⚡ Streaming-Agent",
    description="Antworten erscheinen schrittweise – Chunk für Chunk.",
    examples=["Was ist 2 hoch 10?", "Berechne 1.000 × 365.", "12% von 850 Euro?"],
    type="messages",
)
demo_stream.launch(quiet=True)

# 5 | Human-in-the-Loop UI
---

Das HITL-Muster aus *Human-in-the-Loop* bekommt hier eine echte Benutzeroberfläche.
Der Agent pausiert vor jedem Tool-Aufruf — der Nutzer entscheidet **Approve** oder **Reject**.

```
┌──────────────────────────────────────────────────┐
│  🙋 Human-in-the-Loop Agent                      │
│                                                  │
│  User:  Was ist 15% von 240?                     │
│  Agent: ⏸️ Tool-Aufruf erkannt:                  │
│         berechne_prozent(wert=240, prozent=15)   │
│                                                  │
│  [✅ Genehmigen]   [❌ Ablehnen]                 │
│                                                  │
│  Agent: ✅ Ausgeführt → 36.0                     │
│         15% von 240 Euro sind 36 Euro.           │
└──────────────────────────────────────────────────┘
```

> **Implementierungsansatz:** Das LLM ruft `llm.bind_tools().invoke()` auf.
> Bei einem Tool-Call wird die Ausführung pausiert und dem Nutzer angezeigt.
> Erst nach Genehmigung wird das Tool ausgeführt und das LLM für die
> finale Antwort erneut aufgerufen.

In [ ]:
from IPython.display import Image as IPImage, display
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

# HITL-Agent mit Checkpoint und Interrupt vor Tool-Ausführung
hitl_agent = create_react_agent(
    llm,
    tools=tools_liste,
    checkpointer=MemorySaver(),
    interrupt_before=["tools"],
)

try:
    display(IPImage(hitl_agent.get_graph(xray=True).draw_mermaid_png()))
except Exception as e:
    print(f"(Graph-Visualisierung: {e})")

print("✅ HITL-Agent mit interrupt_before=['tools'] kompiliert")

In [ ]:
import gradio as gr

# Sitzungsstatus: ausstehender Tool-Call und Thread-Konfiguration
sitzung = {
    "thread": {"configurable": {"thread_id": "hitl-demo"}},
    "ausstehend": None,
}

def sende_anfrage(nachricht: str, verlauf: list):
    '''Startet den Agenten. Bei Tool-Call pausiert er und zeigt Approve/Reject.'''
    verlauf = verlauf or []
    verlauf.append({"role": "user", "content": nachricht})

    # LLM aufrufen (ohne Tool-Ausführung)
    llm_mit_tools = llm.bind_tools(tools_liste)
    response = llm_mit_tools.invoke([HumanMessage(nachricht)])

    if response.tool_calls:
        tc = response.tool_calls[0]
        sitzung["ausstehend"] = {"call": tc, "nachrichten": [HumanMessage(nachricht), response]}
        args_str = ", ".join(f"{k}={v}" for k, v in tc["args"].items())
        info = f"Tool: **{tc['name']}**\nArgumente: `{args_str}`"
        verlauf.append({"role": "assistant",
                        "content": f"⏸️ Tool-Aufruf erkannt — bitte bestätigen:\n\n{info}"})
        return "", verlauf, gr.update(visible=True)
    else:
        sitzung["ausstehend"] = None
        verlauf.append({"role": "assistant", "content": response.content})
        return "", verlauf, gr.update(visible=False)

def genehmige(verlauf: list):
    '''Führt den ausstehenden Tool-Call aus und holt finale Antwort.'''
    pending = sitzung.get("ausstehend")
    if not pending:
        return verlauf, gr.update(visible=False)
    tc  = pending["call"]
    msgs = pending["nachrichten"]
    tools_map = {t.name: t for t in tools_liste}
    ergebnis  = tools_map[tc["name"]].invoke(tc["args"])
    msgs.append(ToolMessage(content=str(ergebnis), tool_call_id=tc["id"]))
    llm_mit_tools = llm.bind_tools(tools_liste)
    abschluss = llm_mit_tools.invoke(msgs)
    verlauf.append({"role": "assistant",
                    "content": f"✅ Tool ausgeführt → {ergebnis}\n\n{abschluss.content}"})
    sitzung["ausstehend"] = None
    return verlauf, gr.update(visible=False)

def lehne_ab(verlauf: list):
    '''Lehnt den ausstehenden Tool-Call ab.'''
    sitzung["ausstehend"] = None
    verlauf.append({"role": "assistant",
                    "content": "❌ Tool-Aufruf abgelehnt. Keine Berechnung durchgeführt."})
    return verlauf, gr.update(visible=False)

with gr.Blocks(title="HITL Agent", theme=gr.themes.Soft()) as demo_hitl:
    gr.Markdown("## 🙋 Human-in-the-Loop Agent\nFreigabe vor jedem Tool-Aufruf.")
    chatbot = gr.Chatbot(type="messages", label="💬 Chat", height=350)
    with gr.Row():
        eingabe    = gr.Textbox(placeholder="Stelle eine Frage...",
                                label="Eingabe", scale=4)
        senden_btn = gr.Button("Senden", variant="primary", scale=1)
    with gr.Row(visible=False) as approval_row:
        approve_btn = gr.Button("✅ Genehmigen", variant="primary")
        reject_btn  = gr.Button("❌ Ablehnen",  variant="stop")

    senden_btn.click(sende_anfrage, [eingabe, chatbot],
                     [eingabe, chatbot, approval_row])
    eingabe.submit(sende_anfrage,   [eingabe, chatbot],
                   [eingabe, chatbot, approval_row])
    approve_btn.click(genehmige, [chatbot], [chatbot, approval_row])
    reject_btn.click(lehne_ab,   [chatbot], [chatbot, approval_row])

demo_hitl.launch(quiet=True)

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M27-Gradio-UI", limit=3, show_steps=True)

# A | Aufgabe
---



Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.



<p><font color='black' size="5">
Gradio UI für eigene Agenten
</font></p>

**Aufgabe 1 — Eigener Agent mit ChatInterface:**
Erstelle einen Agenten mit mindestens 2 eigenen Tools (z.B. Wetter, Währungsrechner, Textanalyse). Wickle ihn in `gr.ChatInterface` mit sinnvollen `examples`.

**Aufgabe 2 — Blocks mit zwei Panels:**
Erweitere das `gr.Blocks`-Beispiel aus K3 um einen zweiten Agenten (z.B. einen Recherche-Agenten). Zeige beide Agenten in der UI und vergleiche ihre Tool-Logs.

**Aufgabe 3 — Streaming + HITL kombinieren:**
Kombiniere Streaming (K4) mit Human-in-the-Loop (K5): Der Agent streamt seine Zwischengedanken, pausiert aber vor kritischen Tool-Calls zur Bestätigung.

> 💡 **Tipp:** `demo.launch(share=True)` erzeugt eine öffentliche URL — ideal zum Teilen in Colab.